In [4]:
import fitz  # PyMuPDF
import pandas as pd
import os
import io
import cv2
import pytesseract
import regex as re
import numpy as np
import rasterio
from rasterio.transform import from_gcps
from rasterio.control import GroundControlPoint
from pathlib import Path

# --- STAGE 1: INTELLIGENT INGESTION ---

def extract_assets(pdf_path, output_folder, min_image_size=600):
    """Extracts text, tables, and high-quality map candidates from the PDF."""
    print("  Stage 1: Ingesting assets...")
    pdf_file = fitz.open(pdf_path)
    text_content = ""
    image_candidates = []

    # Create subdirectories
    (output_folder / "text").mkdir(exist_ok=True)
    (output_folder / "images").mkdir(exist_ok=True)
    (output_folder / "georeferenced").mkdir(exist_ok=True)

    for page in pdf_file:
        text_content += page.get_text()
        image_list = page.get_images(full=True)
        for img_index, img in enumerate(image_list):
            xref = img[0]
            base_image = pdf_file.extract_image(xref)
            image_bytes = base_image["image"]
            
            # Use OpenCV to check image size from bytes
            nparr = np.frombuffer(image_bytes, np.uint8)
            img_np = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
            if img_np is not None and (img_np.shape[0] > min_image_size and img_np.shape[1] > min_image_size):
                image_path = output_folder / "images" / f"map_candidate_p{page.number + 1}_{img_index + 1}.png"
                with open(image_path, "wb") as f:
                    f.write(image_bytes)
                image_candidates.append(image_path)
    
    text_path = output_folder / "text" / "full_text_content.txt"
    with open(text_path, "w", encoding="utf-8") as f:
        f.write(text_content)
        
    print(f"    - Extracted full text and {len(image_candidates)} map candidates.")
    return text_path, image_candidates

# --- STAGE 2: GEOSPATIAL CONTEXT DISCOVERY (NLP) ---

def find_geospatial_context(text_path):
    """Scans text for coordinate patterns and CRS information."""
    print("  Stage 2: Discovering geospatial context from text...")
    with open(text_path, 'r', encoding='utf-8') as f:
        text = f.read()

    # Regex to find DMS (Degrees, Minutes, Seconds) and Decimal Degree coordinates
    # This is a simplified example; a robust solution needs more complex patterns.
    dms_pattern = r'\b(\d{1,3})°\s*(\d{1,2})[\'’´`]\s*(\d{1,2}(?:\.\d+)?)\"\s*([NSEW])'
    dd_pattern = r'\b-?\d{1,3}\.\d{4,}\b'
    
    dms_coords = re.findall(dms_pattern, text)
    dd_coords = re.findall(dd_pattern, text)
    
    print(f"    - Found {len(dms_coords)} potential DMS coordinates.")
    print(f"    - Found {len(dd_coords)} potential Decimal Degree coordinates.")
    
    # Placeholder for CRS/Projection finding (e.g., searching for "UTM", "WGS84", "EPSG")
    crs_info = "CRS info not implemented, assuming EPSG:4326 (WGS84)"
    
    return {"dms": dms_coords, "dd": dd_coords, "crs": crs_info}

# --- STAGE 3: MAP ANALYSIS & OCR (COMPUTER VISION) ---

def ocr_map_corners(image_path, margin_size=150):
    """
    Uses OCR to read text from the corners of a map image, which is where
    coordinates are most likely to be printed.
    """
    print(f"  Stage 3: Analyzing map candidate '{image_path.name}' with OCR...")
    try:
        img = cv2.imread(str(image_path))
        h, w, _ = img.shape
        
        # Define corner regions to scan
        corners = {
            "top_left": img[0:margin_size, 0:margin_size],
            "top_right": img[0:margin_size, w-margin_size:w],
            "bottom_left": img[h-margin_size:h, 0:margin_size],
            "bottom_right": img[h-margin_size:h, w-margin_size:w],
        }
        
        corner_text = {}
        for name, region in corners.items():
            # Pre-process for better OCR: grayscale, thresholding
            gray = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)
            thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
            text = pytesseract.image_to_string(thresh, config='--psm 6').replace("\n", " ").strip()
            corner_text[name] = text
            if text:
                print(f"    - OCR found text in {name}: '{text}'")
        return corner_text

    except Exception as e:
        print(f"    ! Could not perform OCR on {image_path.name}. Error: {e}")
        return {}

# --- STAGE 4: SYNTHESIS & TRANSFORMATION ---

def parse_dms_to_dd(d, m, s, direction):
    """Converts DMS string values to a decimal degree float."""
    dd = float(d) + float(m)/60 + float(s)/3600
    if direction in ('S', 'W'):
        dd *= -1
    return dd
    
def attempt_auto_georeference(image_path, text_context, ocr_results, output_folder):
    """
    The "brain" of the operation. Tries to find 4 corner coordinates
    from the OCR results and create a GeoTIFF. This is a simplified logic.
    A production system would require much more sophisticated parsing.
    """
    print(f"  Stage 4: Attempting to synthesize data for '{image_path.name}'...")
    
    # This is a massive simplification. We're just trying to find one lat/lon pair in OCR text.
    # A real system would need to intelligently parse various formats.
    # Example logic: find pairs like "45°30'00"N" and "7°15'00"E"
    # For now, this logic is a placeholder for a much more complex function.

    # TODO: Implement robust logic to parse OCR text into 4 corner coordinates.
    # This is the hardest part. It involves regex, NLP, and heuristics.
    # For this example, we will create dummy GCPs to demonstrate the process.
    
    print("    - Auto-GCP derivation is highly complex and not fully implemented.")
    print("    - As a demo, we will show how the transformation works if GCPs are found.")
    
    try:
        # --- DUMMY GCPs FOR DEMONSTRATION ---
        # Imagine our advanced OCR and NLP found these corner coordinates:
        # Top-Left: 46.0 N, 7.0 E
        # Top-Right: 46.0 N, 8.0 E
        # Bottom-Left: 45.0 N, 7.0 E
        # Bottom-Right: 45.0 N, 8.0 E
        
        with rasterio.open(image_path) as src:
            h, w = src.height, src.width
            
            # Pixel coordinates (row, col) and corresponding Geo coordinates (x, y)
            gcps = [
                GroundControlPoint(row=0, col=0, x=7.0, y=46.0, id='tl'),         # Top-left
                GroundControlPoint(row=0, col=w, x=8.0, y=46.0, id='tr'),        # Top-right
                GroundControlPoint(row=h, col=0, x=7.0, y=45.0, id='bl'),        # Bottom-left
                GroundControlPoint(row=h, col=w, x=8.0, y=45.0, id='br')         # Bottom-right
            ]
            
            # Calculate the transformation
            transform = from_gcps(gcps)
            
            # Write the new GeoTIFF
            output_tif_path = output_folder / "georeferenced" / f"{image_path.stem}_georeferenced.tif"
            with rasterio.open(
                output_tif_path,
                'w',
                driver='GTiff',
                height=src.height,
                width=src.width,
                count=src.count,
                dtype=src.dtypes[0],
                crs='EPSG:4326',  # Assuming WGS84
                transform=transform,
            ) as dst:
                dst.write(src.read())
            
            print(f"    + SUCCESS (DEMO): Georeferenced file created at '{output_tif_path}'")
            return True

    except Exception as e:
        print(f"    ! Georeferencing failed. Error: {e}")
    
    return False

# --- MAIN ORCHESTRATOR ---

def process_pdf(pdf_path, output_base_folder):
    print(f"\n--- Processing: {pdf_path.name} ---")
    pdf_output_folder = Path(output_base_folder) / pdf_path.stem
    pdf_output_folder.mkdir(exist_ok=True, parents=True)

    # Stage 1
    text_path, image_candidates = extract_assets(pdf_path, pdf_output_folder)
    
    # Stage 2
    text_context = find_geospatial_context(text_path)
    
    if not image_candidates:
        print("--- No map candidates found. Skipping georeferencing. ---")
        return
        
    for image_path in image_candidates:
        # Stage 3
        ocr_results = ocr_map_corners(image_path)
        
        # Stage 4
        attempt_auto_georeference(image_path, text_context, ocr_results, pdf_output_folder)


if __name__ == '__main__':
    input_folder = "pdfs"
    output_folder = "extracted_data_smart"
    
    # Ensure input folder exists
    if not Path(input_folder).exists():
        Path(input_folder).mkdir()
        print(f"Created '{input_folder}' folder. Please add your PDFs there and run again.")
    
    pdf_files = list(Path(input_folder).glob("*.pdf"))
    if not pdf_files:
        print(f"No PDFs found in '{input_folder}'.")
    else:
        for pdf_path in pdf_files:
            process_pdf(pdf_path, output_folder)
        print("\n--- Smart processing complete. ---")


--- Processing: AKH_corporate-presentation.pdf ---
  Stage 1: Ingesting assets...
    - Extracted full text and 18 map candidates.
  Stage 2: Discovering geospatial context from text...
    - Found 0 potential DMS coordinates.
    - Found 0 potential Decimal Degree coordinates.
  Stage 3: Analyzing map candidate 'map_candidate_p1_1.png' with OCR...
    ! Could not perform OCR on map_candidate_p1_1.png. Error: tesseract is not installed or it's not in your PATH. See README file for more information.
  Stage 4: Attempting to synthesize data for 'map_candidate_p1_1.png'...
    - Auto-GCP derivation is highly complex and not fully implemented.
    - As a demo, we will show how the transformation works if GCPs are found.
    + SUCCESS (DEMO): Georeferenced file created at 'extracted_data_smart\AKH_corporate-presentation\georeferenced\map_candidate_p1_1_georeferenced.tif'
  Stage 3: Analyzing map candidate 'map_candidate_p3_2.png' with OCR...
    ! Could not perform OCR on map_candidate_p3_

C:\Users\ataoutaou\AppData\Local\anaconda3\envs\geoscrape\lib\site-packages\rasterio\__init__.py:332: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


    + SUCCESS (DEMO): Georeferenced file created at 'extracted_data_smart\AKH_corporate-presentation\georeferenced\map_candidate_p7_1_georeferenced.tif'
  Stage 3: Analyzing map candidate 'map_candidate_p8_2.png' with OCR...
    ! Could not perform OCR on map_candidate_p8_2.png. Error: tesseract is not installed or it's not in your PATH. See README file for more information.
  Stage 4: Attempting to synthesize data for 'map_candidate_p8_2.png'...
    - Auto-GCP derivation is highly complex and not fully implemented.
    - As a demo, we will show how the transformation works if GCPs are found.
    + SUCCESS (DEMO): Georeferenced file created at 'extracted_data_smart\AKH_corporate-presentation\georeferenced\map_candidate_p8_2_georeferenced.tif'
  Stage 3: Analyzing map candidate 'map_candidate_p9_1.png' with OCR...
    ! Could not perform OCR on map_candidate_p9_1.png. Error: tesseract is not installed or it's not in your PATH. See README file for more information.
  Stage 4: Attempting 

In [5]:
import numpy as np
import cv2
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape, mapping
import json

def vectorize_features_from_geotiff(geotiff_path, output_folder):
    """
    Extracts features based on color from a GeoTIFF and saves them as GeoJSON.
    This example targets red lines, which often represent faults.
    """
    print(f"  -> Vectorizing features for '{geotiff_path.name}'...")
    with rasterio.open(geotiff_path) as src:
        # Read the image data
        image = src.read()
        # Rasterio reads as (bands, height, width), OpenCV wants (height, width, bands)
        image_cv = np.moveaxis(image, 0, -1).copy()

        # --- Define Color Range for Extraction (e.g., for RED faults) ---
        # Convert to HSV color space, which is better for color detection
        hsv = cv2.cvtColor(image_cv, cv2.COLOR_RGB2HSV)
        lower_red = np.array([0, 120, 70])
        upper_red = np.array([10, 255, 255])
        mask1 = cv2.inRange(hsv, lower_red, upper_red)
        
        # Second range for red that wraps around HSV
        lower_red = np.array([170, 120, 70])
        upper_red = np.array([180, 255, 255])
        mask2 = cv2.inRange(hsv, lower_red, upper_red)

        # Combine masks to get all red colors
        mask = mask1 + mask2
        
        if np.count_nonzero(mask) == 0:
            print("    - No features found for the specified color range.")
            return

        # --- Convert Detected Pixels to Vector Shapes ---
        # The 'shapes' function requires a specific data type and transform
        mask = mask.astype(np.uint8)
        feature_shapes = shapes(mask, mask=mask, transform=src.transform)

        # --- Save as GeoJSON ---
        geojson_features = []
        for geom, val in feature_shapes:
            if val == 255: # 255 is the value for features found in our mask
                feature = {
                    "type": "Feature",
                    "geometry": mapping(shape(geom)),
                    "properties": {"type": "fault_line"} # Add attributes
                }
                geojson_features.append(feature)
        
        geojson_collection = {"type": "FeatureCollection", "features": geojson_features}
        output_path = output_folder / "vectorized" / f"{geotiff_path.stem}_faults.geojson"
        output_path.parent.mkdir(exist_ok=True)

        with open(output_path, "w") as f:
            json.dump(geojson_collection, f)
            
        print(f"    + Saved vectorized features to '{output_path}'")

In [2]:
import tabula
import os

# Define the path to your PDF file
pdf_path = r'C:\Users\ataoutaou\pdfs_extraction\pdfs\Gabr 2015.pdf' # <-- IMPORTANT: Change this to your PDF file's name

# Check if the PDF file exists
if not os.path.exists(pdf_path):
    print(f"Error: The file '{pdf_path}' was not found.")
else:
    print(f"Reading tables from '{pdf_path}'...")
    
    # Use tabula to read tables from the PDF
    # pages='all' will search all pages of the document
    # multiple_tables=True allows extracting multiple tables from a single page
    try:
        tables = tabula.read_pdf(pdf_path, pages='all', multiple_tables=True)

        if not tables:
            print("No tables were found in the PDF.")
        else:
            # Create a directory to store the output CSV files
            output_dir = 'extracted_tables'
            if not os.path.exists(output_dir):
                os.makedirs(output_dir)
            
            print(f"Found {len(tables)} tables. Saving them as CSV files in the '{output_dir}' directory...")

            # Loop through all the found tables and save each one as a CSV file
            for i, table in enumerate(tables):
                # The table object is a pandas DataFrame
                output_path = os.path.join(output_dir, f'table_{i + 1}.csv')
                table.to_csv(output_path, index=False)
                print(f" -> Saved table_{i + 1}.csv")
            
            print("\nExtraction complete! ✨")

    except Exception as e:
        print(f"An error occurred: {e}")
        print("Please ensure Java is installed and accessible in your system's PATH.")

Reading tables from 'C:\Users\ataoutaou\pdfs_extraction\pdfs\Gabr 2015.pdf'...
Found 6 tables. Saving them as CSV files in the 'extracted_tables' directory...
 -> Saved table_1.csv
 -> Saved table_2.csv
 -> Saved table_3.csv
 -> Saved table_4.csv
 -> Saved table_5.csv
 -> Saved table_6.csv

Extraction complete! ✨


In [ ]:
import pdfplumber
import pandas as pd
import os

# --- Configuration ---
# 1. Set the path to your PDF file
pdf_path = r'C:\Users\ataoutaou\pdfs_extraction\pdfs\Gabr 2015.pdf' # <-- IMPORTANT: Change this to your PDF's filename

# 2. Set the name of the folder where CSVs will be saved
output_folder = 'pdfplumber_tables'
# --- End of Configuration ---

# Create the output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Check if the PDF file exists before proceeding
if not os.path.exists(pdf_path):
    print(f"Error: The file '{pdf_path}' was not found. Please check the filename.")
else:
    print(f"Processing '{pdf_path}'...")
    # Open the PDF file
    with pdfplumber.open(pdf_path) as pdf:
        table_count = 0
        # Loop through each page of the PDF
        for i, page in enumerate(pdf.pages):
            # The .extract_tables() method finds and extracts all tables on a page
            tables = page.extract_tables()
            
            # Loop through all tables found on the current page
            for j, table_data in enumerate(tables):
                if table_data:  # Check if the table_data is not empty
                    table_count += 1
                    # The first row is usually the header
                    header = table_data[0]
                    # The rest of the data
                    data = table_data[1:]
                    
                    # Convert the list of lists into a pandas DataFrame
                    df = pd.DataFrame(data, columns=header)
                    
                    # Define the output filename
                    output_filename = os.path.join(output_folder, f'page_{i+1}_table_{j+1}.csv')
                    
                    # Save the DataFrame to a CSV file
                    df.to_csv(output_filename, index=False)
                    print(f"✅ Saved: {output_filename}")

    if table_count == 0:
        print("No tables were found in the document.")
    else:
        print(f"\n✨ Done! Extracted {table_count} tables into the '{output_folder}' folder.")

Processing 'C:\Users\ataoutaou\pdfs_extraction\pdfs\Gabr 2015.pdf'...


# Gorbid

In [15]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import os

# --- Configuration ---
pdf_path =  r'C:\Users\ataoutaou\pdfs_extraction\pdfs\Augustin2017.pdf'
output_folder = 'grobid_tables_arnaud_2017'
grobid_url = "http://localhost:8070/api/processFulltextDocument"
# --- End of Configuration ---

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

print(f"Sending '{pdf_path}' to the Grobid server...")

try:
    with open(pdf_path, 'rb') as f:
        response = requests.post(grobid_url, files={'input': f})

    if response.status_code == 200:
        print("✅ Successfully processed by Grobid. Parsing result...")
        
        soup = BeautifulSoup(response.text, 'lxml-xml')
        tables = soup.find_all('figure', {'type': 'table'})
        
        if not tables:
            print("No tables were found by Grobid.")
        else:
            print(f"Found {len(tables)} tables. Attempting to convert to CSV...")
            successful_saves = 0
            for i, table_xml in enumerate(tables):
                all_rows = table_xml.find_all('row')
                
                if not all_rows:
                    print(f" -> Skipped table {i+1} because no rows could be parsed.")
                    continue

                header = []
                table_head = table_xml.find('head')
                if table_head and table_head.find('row'):
                    header = [cell.text for cell in table_head.find('row').find_all('cell')]
                
                data_rows_raw = [row for row in all_rows]
                
                if not header and data_rows_raw:
                    header = [cell.text for cell in data_rows_raw.pop(0).find_all('cell')]
                
                if header and data_rows_raw and [cell.text for cell in data_rows_raw[0].find_all('cell')] == header:
                    data_rows_raw.pop(0)

                if not header and data_rows_raw:
                    num_columns = max(len(row.find_all('cell')) for row in data_rows_raw)
                    header = [f'Column_{k+1}' for k in range(num_columns)]
                
                # --- START OF FIX ---
                # Normalize the data: ensure every row has the same number of columns as the header
                num_columns = len(header)
                data = []
                for row in data_rows_raw:
                    cells = [cell.text for cell in row.find_all('cell')]
                    # Trim rows that are too long, and pad rows that are too short
                    normalized_row = (cells + [''] * num_columns)[:num_columns]
                    data.append(normalized_row)
                # --- END OF FIX ---
                
                if not data:
                    print(f" -> Skipped table {i+1} as it contained no data rows after parsing.")
                    continue
                
                df = pd.DataFrame(data, columns=header)
                output_path = os.path.join(output_folder, f"grobid_table_{i+1}.csv")
                df.to_csv(output_path, index=False)
                print(f" -> Saved {output_path}")
                successful_saves += 1
            
            print(f"\n✨ Done! Successfully extracted {successful_saves} tables.")
            
    else:
        print(f"❌ Grobid server returned an error: {response.status_code} - {response.text}")

except FileNotFoundError:
    print(f"❌ File Not Found Error: The file '{pdf_path}' was not found.")
except requests.exceptions.ConnectionError:
    print("❌ Connection Error: Could not connect to the Grobid server. Is the Docker container running?")

Sending 'C:\Users\ataoutaou\pdfs_extraction\pdfs\Augustin2017.pdf' to the Grobid server...
✅ Successfully processed by Grobid. Parsing result...
Found 9 tables. Attempting to convert to CSV...
 -> Skipped table 1 because no rows could be parsed.
 -> Saved grobid_tables_arnaud_2017\grobid_table_2.csv
 -> Saved grobid_tables_arnaud_2017\grobid_table_3.csv
 -> Saved grobid_tables_arnaud_2017\grobid_table_4.csv
 -> Saved grobid_tables_arnaud_2017\grobid_table_5.csv
 -> Saved grobid_tables_arnaud_2017\grobid_table_6.csv
 -> Saved grobid_tables_arnaud_2017\grobid_table_7.csv
 -> Saved grobid_tables_arnaud_2017\grobid_table_8.csv
 -> Saved grobid_tables_arnaud_2017\grobid_table_9.csv

✨ Done! Successfully extracted 8 tables.


# vision-based

In [4]:
# You must first install the required packages and Tesseract OCR
# pip install "unstructured[local-inference]"
# pip install "detectron2@git+https://github.com/facebookresearch/detectron2.git@v0.6#egg=detectron2"
from unstructured.partition.pdf import partition_pdf
import pandas as pd

elements = partition_pdf(
    r'C:\Users\ataoutaou\pdfs_extraction\pdfs\Augustin2017.pdf',
    strategy="hi_res", # This activates the vision-based model
    infer_table_structure=True
)

tables = [el for el in elements if el.category == "Table"]

for i, table_element in enumerate(tables):
    html_string = table_element.metadata.text_as_html
    df = pd.read_html(html_string)[0]
    df.to_csv(f'vision_extracted_table_{i+1}.csv', index=False)
    print(f"✅ Saved vision-based table {i+1}")

ModuleNotFoundError: No module named 'unstructured'

# Geometric approach 

In [5]:
import pdfplumber
import pandas as pd
import os

pdf_path = r'C:\Users\ataoutaou\pdfs_extraction\pdfs\Augustin2017.pdf' # <-- Change to your filename
output_folder = 'pdfplumber_annex_tables'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Focus only on the last few pages where annexes are
with pdfplumber.open(pdf_path) as pdf:
    # Example: To process the last 3 pages
    # Adjust the number based on your document
    start_page = len(pdf.pages) - 3 
    
    for i, page_num in enumerate(range(start_page, len(pdf.pages))):
        page = pdf.pages[page_num]
        
        # Use tuned settings for tables that might not have visible lines
        tables = page.extract_tables({
            "vertical_strategy": "text",
            "horizontal_strategy": "lines",
        })
        
        for j, table_data in enumerate(tables):
            df = pd.DataFrame(table_data[1:], columns=table_data[0])
            df.to_csv(os.path.join(output_folder, f'page_{page_num+1}_table_{j+1}.csv'), index=False)
            print(f"✅ Saved page {page_num+1}, table {j+1}")

✅ Saved page 92, table 1


# Gorbid configuration

In [17]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import os
import re
import logging
import pdfplumber
from typing import Optional, List, Dict

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class HybridTableExtractor:
    def __init__(self, grobid_url: str = "http://localhost:8070/api/processFulltextDocument", 
                 output_folder: str = 'hybrid_extracted_tables'):
        self.grobid_url = grobid_url
        self.output_folder = output_folder
        self.create_output_folder()

    def create_output_folder(self):
        if not os.path.exists(self.output_folder):
            os.makedirs(self.output_folder)

    def _get_table_locations_from_grobid(self, pdf_path: str) -> List[Dict]:
        """Uses Grobid to find the page and coordinates of every table."""
        logger.info("Step 1: Asking Grobid to find table locations...")
        locations = []
        try:
            with open(pdf_path, 'rb') as f:
                response = requests.post(
                    self.grobid_url, 
                    files={'input': f}, 
                    data={'teiCoordinates': 'figure'}
                )
            
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'lxml-xml')
                tables = soup.find_all('figure', {'type': 'table'})
                logger.info(f"Grobid found {len(tables)} potential tables.")
                
                for table_xml in tables:
                    coords_str = table_xml.get('coords')
                    if coords_str:
                        parts = coords_str.split(',')
                        page_num = int(parts[0])
                        x0, y0, w, h = [float(p) for p in parts[1:]]
                        bounding_box = (x0, y0, x0 + w, y0 + h)
                        locations.append({'page': page_num, 'bbox': bounding_box})
                return locations
            else:
                logger.error(f"Grobid server returned error: {response.status_code}")
                return []
        except Exception as e:
            logger.error(f"Error communicating with Grobid: {e}")
            return []

    def _extract_data_with_pdfplumber(self, pdf_path: str, locations: List[Dict]) -> List[pd.DataFrame]:
        """Uses pdfplumber to extract data from the specific locations Grobid found."""
        logger.info("Step 2: Using pdfplumber to extract data from locations...")
        extracted_tables = []
        with pdfplumber.open(pdf_path) as pdf:
            for i, loc in enumerate(locations):
                page_num = loc['page']
                bbox = loc['bbox']
                
                if page_num > len(pdf.pages):
                    logger.warning(f"Grobid reported a table on page {page_num}, but PDF only has {len(pdf.pages)} pages.")
                    continue
                
                page = pdf.pages[page_num - 1]
                table_area = page.crop(bbox)
                
                # --- START OF FIX ---
                # Define a strategy for borderless tables
                table_settings = {
                    "vertical_strategy": "text",
                    "horizontal_strategy": "text",
                }
                
                # Apply the new settings
                table_data = table_area.extract_table(table_settings=table_settings)
                # --- END OF FIX ---
                
                if table_data:
                    df = pd.DataFrame(table_data[1:], columns=table_data[0])
                    extracted_tables.append(df)
                    logger.info(f"Successfully extracted table from page {page_num}")
                else:
                    logger.warning(f"pdfplumber failed to extract table from page {page_num} at bbox {bbox}")
                    # You can uncomment the line below to save a picture of the failed area for debugging
                    # table_area.to_image().save(f"debug_table_{i}.png")
        return extracted_tables

    def extract_tables(self, pdf_path: str):
        locations = self._get_table_locations_from_grobid(pdf_path)
        
        if not locations:
            logger.warning("Grobid found no table locations. Stopping.")
            return

        final_tables = self._extract_data_with_pdfplumber(pdf_path, locations)

        if not final_tables:
            logger.warning("pdfplumber could not extract any tables from the locations Grobid provided.")
            return
            
        for i, df in enumerate(final_tables):
            df.dropna(axis=0, how='all', inplace=True)
            df.dropna(axis=1, how='all', inplace=True)
            output_path = os.path.join(self.output_folder, f"hybrid_table_{i+1}.csv")
            df.to_csv(output_path, index=False)
            logger.info(f"✅ Saved final table to {output_path}")

        logger.info(f"✨ Extraction complete. Saved {len(final_tables)} tables.")

# ==============================================================================
# HOW TO USE
# ==============================================================================
pdf_path =  r'C:\Users\ataoutaou\pdfs_extraction\pdfs\Augustin2017.pdf'  # <-- IMPORTANT: Set your PDF path here

extractor = HybridTableExtractor()
extractor.extract_tables(pdf_path)

2025-09-04 10:32:56,930 - INFO - Step 1: Asking Grobid to find table locations...
2025-09-04 10:33:03,649 - INFO - Grobid found 9 potential tables.
2025-09-04 10:33:03,653 - INFO - Step 2: Using pdfplumber to extract data from locations...
2025-09-04 10:33:03,960 - INFO - Successfully extracted table from page 58
2025-09-04 10:33:04,178 - INFO - Successfully extracted table from page 81
2025-09-04 10:33:04,398 - INFO - Successfully extracted table from page 82
2025-09-04 10:33:04,779 - INFO - Successfully extracted table from page 84
2025-09-04 10:33:05,221 - INFO - Successfully extracted table from page 85
2025-09-04 10:33:05,362 - INFO - Successfully extracted table from page 86
2025-09-04 10:33:05,655 - INFO - Successfully extracted table from page 87
2025-09-04 10:33:05,928 - INFO - Successfully extracted table from page 88
2025-09-04 10:33:06,190 - INFO - Successfully extracted table from page 89
2025-09-04 10:33:06,198 - INFO - ✅ Saved final table to hybrid_extracted_tables\hybri

In [21]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import os
import re
import logging
import pdfplumber
from typing import Optional, List, Dict

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class ExpertTableExtractor:
    def __init__(self, grobid_url: str = "http://localhost:8070/api/processFulltextDocument", 
                 output_folder: str = 'expert_extracted_tables'):
        self.grobid_url = grobid_url
        self.output_folder = output_folder
        self.create_output_folder()

    def create_output_folder(self):
        if not os.path.exists(self.output_folder):
            os.makedirs(self.output_folder)

    def _get_table_locations_from_grobid(self, pdf_path: str) -> List[Dict]:
        # This function is the same as the previous version
        logger.info("Step 1: Asking Grobid to find table locations...")
        locations = []
        try:
            with open(pdf_path, 'rb') as f:
                response = requests.post(
                    self.grobid_url, files={'input': f}, data={'teiCoordinates': 'figure'}
                )
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'lxml-xml')
                tables = soup.find_all('figure', {'type': 'table'})
                logger.info(f"Grobid found {len(tables)} potential tables.")
                for table_xml in tables:
                    coords_str = table_xml.get('coords')
                    if coords_str:
                        parts = coords_str.split(',')
                        page_num = int(parts[0])
                        x0, y0, w, h = [float(p) for p in parts[1:]]
                        bounding_box = (x0, y0, x0 + w, y0 + h)
                        locations.append({'page': page_num, 'bbox': bounding_box, 'id': len(locations)})
                return locations
            else:
                logger.error(f"Grobid server returned error: {response.status_code}")
        except Exception as e:
            logger.error(f"Error communicating with Grobid: {e}")
        return []

    def _is_valid_table(self, df: Optional[pd.DataFrame]) -> bool:
        if df is None or df.empty: return False
        return df.shape[0] > 1 and df.shape[1] > 1

    def _headers_are_similar(self, h1: list, h2: list, threshold: float = 0.8) -> bool:
        if len(h1) != len(h2): return False
        matches = sum(1 for c1, c2 in zip(h1, h2) if c1 == c2)
        return (matches / len(h1)) >= threshold

    # --- START OF FIX ---
    def _deduplicate_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Ensures all column names in a DataFrame are unique."""
        cols = pd.Series(df.columns)
        for dup in cols[cols.duplicated()].unique():
            cols[cols[cols == dup].index.values.tolist()] = [dup + '.' + str(i) if i != 0 else dup for i in range(sum(cols == dup))]
        df.columns = cols
        return df
    # --- END OF FIX ---

    def extract_and_stitch_tables(self, pdf_path: str):
        locations = self._get_table_locations_from_grobid(pdf_path)
        if not locations:
            logger.warning("Grobid found no table locations. Stopping.")
            return

        table_settings = {
            "vertical_strategy": "text", "horizontal_strategy": "text",
            "snap_x_tolerance": 4, "join_tolerance": 4,
        }
        
        parsed_pieces = []
        with pdfplumber.open(pdf_path) as pdf:
            for loc in locations:
                page = pdf.pages[loc['page'] - 1]
                table_area = page.crop(loc['bbox'])
                table_data = table_area.extract_table(table_settings=table_settings)
                
                if table_data:
                    df = pd.DataFrame(table_data[1:], columns=table_data[0])
                    if self._is_valid_table(df):
                        # Apply the fix here to ensure every piece has unique columns
                        df = self._deduplicate_columns(df)
                        parsed_pieces.append({'df': df, 'page': loc['page'], 'id': loc['id']})
                        logger.info(f"Successfully extracted piece from page {loc['page']}")
                    else:
                        logger.warning(f"Filtered out a false positive on page {loc['page']}")

        if not parsed_pieces:
            logger.warning("No valid table pieces were extracted after filtering.")
            return

        sorted_tables = sorted(parsed_pieces, key=lambda x: (x['page'], x['id']))
        final_tables = []
        processed_indices = set()

        for i in range(len(sorted_tables) - 1):
            if i in processed_indices: continue
            current = sorted_tables[i]
            next_table = sorted_tables[i+1]
            is_on_next_page = (next_table['page'] == current['page'] + 1)
            headers_similar = self._headers_are_similar(list(current['df'].columns), list(next_table['df'].columns))

            if is_on_next_page and headers_similar:
                logger.info(f"Stitching table from page {current['page']} with page {next_table['page']}")
                # The columns are already de-duplicated, so concat will work
                stitched_df = pd.concat([current['df'], next_table['df']], ignore_index=True)
                next_table['df'] = stitched_df
                processed_indices.add(i)
        
        for i, table in enumerate(sorted_tables):
            if i not in processed_indices:
                final_tables.append(table['df'])

        for i, df in enumerate(final_tables):
            df.dropna(axis=0, how='all', inplace=True)
            df.dropna(axis=1, how='all', inplace=True)
            output_path = os.path.join(self.output_folder, f"expert_table_{i+1}.csv")
            df.to_csv(output_path, index=False)
            logger.info(f"✅ Saved final table to {output_path}")

        logger.info(f"✨ Extraction complete. Saved {len(final_tables)} final tables.")

# ==============================================================================
# HOW TO USE
# ==============================================================================
pdf_path = r'C:\Users\ataoutaou\pdfs_extraction\pdfs\Augustin2017.pdf' # <-- IMPORTANT: Set your PDF path here

extractor = ExpertTableExtractor()
extractor.extract_and_stitch_tables(pdf_path)

2025-09-04 10:52:47,421 - INFO - Step 1: Asking Grobid to find table locations...
2025-09-04 10:52:52,207 - INFO - Grobid found 9 potential tables.
2025-09-04 10:52:52,558 - INFO - Successfully extracted piece from page 58
2025-09-04 10:52:52,772 - INFO - Successfully extracted piece from page 81
2025-09-04 10:52:52,995 - INFO - Successfully extracted piece from page 82
2025-09-04 10:52:53,348 - INFO - Successfully extracted piece from page 84
2025-09-04 10:52:53,765 - INFO - Successfully extracted piece from page 85
2025-09-04 10:52:53,961 - INFO - Successfully extracted piece from page 86
2025-09-04 10:52:54,300 - INFO - Successfully extracted piece from page 87
2025-09-04 10:52:54,625 - INFO - Successfully extracted piece from page 88
2025-09-04 10:52:54,982 - INFO - Successfully extracted piece from page 89
2025-09-04 10:52:54,984 - INFO - Stitching table from page 85 with page 86
2025-09-04 10:52:54,995 - INFO - Stitching table from page 87 with page 88
2025-09-04 10:52:54,995 - I

In [19]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import os
import re
import logging
import pdfplumber
from typing import Optional, List, Dict

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class HybridTableExtractor:
    def __init__(self, grobid_url: str = "http://localhost:8070/api/processFulltextDocument", 
                 output_folder: str = 'hybrid_extracted_tables'):
        self.grobid_url = grobid_url
        self.output_folder = output_folder
        self.create_output_folder()

    def create_output_folder(self):
        if not os.path.exists(self.output_folder):
            os.makedirs(self.output_folder)

    # --- THIS METHOD WAS MISSING ---
    def _get_table_locations_from_grobid(self, pdf_path: str) -> List[Dict]:
        """Uses Grobid to find the page and coordinates of every table."""
        logger.info("Step 1: Asking Grobid to find table locations...")
        locations = []
        try:
            with open(pdf_path, 'rb') as f:
                response = requests.post(
                    self.grobid_url, 
                    files={'input': f}, 
                    data={'teiCoordinates': 'figure'}
                )
            
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'lxml-xml')
                tables = soup.find_all('figure', {'type': 'table'})
                logger.info(f"Grobid found {len(tables)} potential tables.")
                
                for table_xml in tables:
                    coords_str = table_xml.get('coords')
                    if coords_str:
                        parts = coords_str.split(',')
                        page_num = int(parts[0])
                        x0, y0, w, h = [float(p) for p in parts[1:]]
                        bounding_box = (x0, y0, x0 + w, y0 + h)
                        locations.append({'page': page_num, 'bbox': bounding_box})
                return locations
            else:
                logger.error(f"Grobid server returned error: {response.status_code}")
                return []
        except Exception as e:
            logger.error(f"Error communicating with Grobid: {e}")
            return []
    
    # --- THIS IS THE NEW DEBUGGING METHOD ---
    def extract_tables_for_debugging(self, pdf_path: str):
        """Saves an image of each table area Grobid finds."""
        locations = self._get_table_locations_from_grobid(pdf_path)
        
        if not locations:
            logger.warning("Grobid found no table locations.")
            return

        debug_folder = "table_debug_images"
        if not os.path.exists(debug_folder):
            os.makedirs(debug_folder)
        
        with pdfplumber.open(pdf_path) as pdf:
            for i, loc in enumerate(locations):
                page_num = loc['page']
                bbox = loc['bbox']
                
                if page_num > len(pdf.pages):
                    continue
                
                page = pdf.pages[page_num - 1]
                table_area = page.crop(bbox)
                
                image_path = os.path.join(debug_folder, f"debug_page_{page_num}_table_{i}.png")
                table_area.to_image(resolution=200).save(image_path)
                logger.info(f"✅ Saved debug image to {image_path}")

# ==============================================================================
# HOW TO USE
# ==============================================================================
pdf_path = r'C:\Users\ataoutaou\pdfs_extraction\pdfs\Augustin2017.pdf' #<-- SET YOUR PATH

debug_extractor = HybridTableExtractor()
debug_extractor.extract_tables_for_debugging(pdf_path)

2025-09-04 10:39:54,429 - INFO - Step 1: Asking Grobid to find table locations...
2025-09-04 10:40:00,782 - INFO - Grobid found 9 potential tables.
2025-09-04 10:40:01,274 - INFO - ✅ Saved debug image to table_debug_images\debug_page_58_table_0.png
2025-09-04 10:40:01,341 - INFO - ✅ Saved debug image to table_debug_images\debug_page_81_table_1.png
2025-09-04 10:40:01,405 - INFO - ✅ Saved debug image to table_debug_images\debug_page_82_table_2.png
2025-09-04 10:40:01,494 - INFO - ✅ Saved debug image to table_debug_images\debug_page_84_table_3.png
2025-09-04 10:40:01,585 - INFO - ✅ Saved debug image to table_debug_images\debug_page_85_table_4.png
2025-09-04 10:40:01,650 - INFO - ✅ Saved debug image to table_debug_images\debug_page_86_table_5.png
2025-09-04 10:40:01,727 - INFO - ✅ Saved debug image to table_debug_images\debug_page_87_table_6.png
2025-09-04 10:40:01,800 - INFO - ✅ Saved debug image to table_debug_images\debug_page_88_table_7.png
2025-09-04 10:40:01,886 - INFO - ✅ Saved deb

### vision based script 

In [ ]:
import pandas as pd
import os
from unstructured.partition.pdf import partition_pdf

# --- Configuration ---
pdf_path =   r'C:\Users\ataoutaou\pdfs_extraction\pdfs\Augustin2017.pdf' # <-- IMPORTANT: Set your PDF path
output_folder = 'vision_extracted_tables'
stitched_folder = 'vision_stitched_tables'
# --- End of Configuration ---

if not os.path.exists(output_folder):
    os.makedirs(output_folder)
if not os.path.exists(stitched_folder):
    os.makedirs(stitched_folder)

print(f"Processing '{pdf_path}' with the vision model...")
try:
    elements = partition_pdf(
        pdf_path,
        strategy="hi_res", # This activates the vision-based model
        infer_table_structure=True
    )

    table_elements = [el for el in elements if el.category == "Table"]
    parsed_pieces = []

    for i, table_el in enumerate(table_elements):
        page_number = table_el.metadata.page_number
        try:
            # The header=0 argument tells pandas to use the first row as the header
            df = pd.read_html(table_el.metadata.text_as_html, header=0)[0]
            parsed_pieces.append({'df': df, 'page': page_number})
            print(f"✅ Extracted table piece from page {page_number}")
        except Exception as e:
            print(f"⚠️ Could not parse table {i} on page {page_number}: {e}")

    # --- Stitching Logic ---
    print("\nAttempting to stitch table pieces...")
    if not parsed_pieces:
        print("No table pieces were extracted to stitch.")
    else:
        sorted_tables = sorted(parsed_pieces, key=lambda x: x['page'])
        final_tables = []
        # Start with the first table
        current_stitched_table = sorted_tables[0]['df']

        for i in range(len(sorted_tables) - 1):
            next_table_info = sorted_tables[i+1]
            is_on_next_page = (next_table_info['page'] == sorted_tables[i]['page'] + 1)
            # Check if column counts are the same, a good indicator of a continued table
            cols_match = (len(current_stitched_table.columns) == len(next_table_info['df'].columns))

            if is_on_next_page and cols_match:
                print(f"Stitching table from page {sorted_tables[i]['page']} with page {next_table_info['page']}")
                next_df = next_table_info['df']
                # Ensure the column names from the first piece are used
                next_df.columns = current_stitched_table.columns
                current_stitched_table = pd.concat([current_stitched_table, next_df], ignore_index=True)
            else:
                final_tables.append(current_stitched_table)
                current_stitched_table = next_table_info['df']
        
        # Add the last table to the list
        final_tables.append(current_stitched_table)

        # Save the final, stitched tables
        for i, final_df in enumerate(final_tables):
            output_path = os.path.join(stitched_folder, f'final_table_{i+1}.csv')
            final_df.to_csv(output_path, index=False)
            print(f"✅ Saved final stitched table to {output_path}")

        print(f"\n✨ Extraction complete. Saved {len(final_tables)} final tables.")

except Exception as e:
    print(f"\n❌ An error occurred during processing: {e}")